**Fine-Tuning BERT for POS Tagging &** **Chunking**

Task 1: Dataset Selection

Dataset Chosen:

Universal Dependencies (UD) → for POS Tagging
CoNLL-2003 → for Chunking (NER-style chunk labels)

Label Categories:

POS Tags (UD example)
NOUN, VERB, ADJ, ADV, PRON, DET, ADP, AUX, CCONJ, NUM, PART, INTJ, PROPN, PUNCT
Chunk Tags (CoNLL format)
B-NP (Beginning of Noun Phrase)
I-NP (Inside Noun Phrase)
B-VP (Verb Phrase)
I-VP
B-PP (Prepositional Phrase)
O (Outside)

Task 2: Data Preprocessing

Key Steps:
Tokenization using BERT tokenizer

Align labels with subwords

Assign -100 for ignored token

In [ ]:
#install
!pip install transformers datasets seqeval

In [30]:
#Import Libraries
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer
from datasets import Dataset
import numpy as np

In [ ]:
#Create Dataset
data = [
    {"tokens": ["John", "works", "at", "Google"], "labels": ["NOUN", "VERB", "ADP", "NOUN"]},
    {"tokens": ["Mary", "lives", "in", "London"], "labels": ["NOUN", "VERB", "ADP", "NOUN"]},
    {"tokens": ["He", "is", "a", "doctor"], "labels": ["PRON", "VERB", "DET", "NOUN"]},
]

dataset = Dataset.from_list(data)
print(dataset)

In [32]:
#Label Mapping
label_list = list(set(label for row in data for label in row["labels"]))

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

In [33]:
#Tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

In [ ]:
#Tokenization + Alignment
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label[word_idx]])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

Task 3: Model Setup

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

Task 4: Training

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    num_train_epochs=3,
    logging_dir="./logs",
)
from datasets import load_metric
metric = load_metric("seqeval")
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id2label[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,
    compute_metrics=compute_metrics
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,
    compute_metrics=compute_metrics
)

Task 5: Evaluation

In [ ]:
trainer.train()
trainer.predict(tokenized_dataset)

Task 6: Inference

In [ ]:
from transformers import pipeline

nlp = pipeline("token-classification", model=model, tokenizer=tokenizer)

sentence = "John works at Google in California"
result = nlp(sentence)

for r in result:
    print(r)